## Mixture of Agents
Here we can use the Python SDK to develop a mixture of agents, then save the agent to a config.yaml and run it from there.

In [7]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [8]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

In [ ]:
system_prompt = """
Answer the following questions as best you can. You may communicate and collaborate with various experts to answer the
questions:

{tools}

You may respond in one of two formats.
Use the following format exactly to communicate with an expert:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action (if there is no required input, include "Action Input: None")
Observation: wait for the expert to respond, do not assume the expert's response

... (this Thought/Action/Action Input/Observation can repeat N times.)
Use the following format once you have the final answer:

Thought: I now know the final answer
Final Answer: the final answer to the original input question
"""

In [ ]:
from nat.llm.nim_llm import NIMModelConfig
from nat.plugins.langchain.tools.code_generation_tool import CodeGenerationTool
from nat.plugins.langchain.tools.code_generation_tool import code_generation_tool
from nat.plugins.langchain.tools.wikipedia_search import WikiSearchToolConfig
from nat.plugins.langchain.tools.wikipedia_search import wiki_search
from nat.tool.datetime_tools import CurrentTimeToolConfig
from nat.tool.datetime_tools import current_datetime
from nat.utils.sdk.nat_agent import NatReactAgent
from nat.utils.sdk.nat_agent import NatToolCallingAgent
from nat.utils.sdk.nat_llm import NatLLM
from nat.utils.sdk.nat_tool import NatTool
from nat.utils.sdk.nat_tool_group import NatToolGroup
from nat_simple_calculator.register import CalculatorToolConfig
from nat_simple_calculator.register import calculator

agent_orchestrator_llm = NatLLM(
    config=NIMModelConfig(
        model_name="nvdev/meta/llama-3.1-405b-instruct",
        temperature=0.2,
        max_tokens=250,
    ),
    name="agent_orchestrator",
)
agent_executor_llm = NatLLM(
    config=NIMModelConfig(
        model_name="nvdev/meta/llama-3.3-70b-instruct",
        temperature=0,
        max_tokens=250,
    ),
    name="agent_executor",
)
calculator_tool_group = NatToolGroup(
    config=CalculatorToolConfig(),
    tool_group=calculator,
    name="calculator",
)

# Define a list of tools
wikipedia_search_tool = NatTool(
    config=WikiSearchToolConfig(max_results=3),
    function=wiki_search,
    name="wiki_search",
)
current_time_tool = NatTool(
    config=CurrentTimeToolConfig(),
    function=current_datetime,
    name="current_datetime",
)
generate_code_tool = NatTool(
    config=CodeGenerationTool(
        programming_language="Python",
        description=
        "Useful to generate Python code. For any questions about code generation, you must only use this tool!",
        llm_name=agent_orchestrator_llm.llm_name,
        verbose=True,
    ),
    function=code_generation_tool,
    name="code_generation",
)

# Define a list of agents
math_agent = NatToolCallingAgent(tool_groups=[calculator_tool_group],
                                 llm=agent_executor_llm,
                                 verbose=True,
                                 handle_tool_errors=True,
                                 description="Useful for performing simple mathematical calculations.",
                                 agent_name="math_agent")
internet_agent = NatToolCallingAgent(tools=[wikipedia_search_tool, current_time_tool],
                                     llm=agent_executor_llm,
                                     verbose=True,
                                     handle_tool_errors=True,
                                     description="Useful for performing simple internet searches.",
                                     agent_name="internet_agent")

agent = NatReactAgent(tools=[math_agent, internet_agent, generate_code_tool],
                      llm=agent_orchestrator_llm,
                      verbose=True,
                      parse_agent_response_max_retries=2,
                      system_prompt=system_prompt)

In [11]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
agent.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  math_agent:
    _type: tool_calling_agent
    llm_name: agent_executor
    verbose: true
    description: Useful for performing simple mathematical calculations.
    tool_names:
    - calculator
    handle_tool_errors: true
  internet_agent:
    _type: tool_calling_agent
    llm_name: agent_executor
    verbose: true
    description: Useful for performing simple internet searches.
    tool_names:
    - wiki_search
    - current_datetime
    handle_tool_errors: true
  wiki_search:
    _type: wiki_search
    max_results: 3
  current_datetime:
    _type: current_datetime
  code_generation:
    _type: code_generation
    llm_name: agent_orchestrator
    verbose: true
    programming_language: Python
    description: |-
      Useful to generate Python code. For any questions about code generation, you must only use this
      tool!

function_groups:
  calculator:
    _type: calculator

llms:
  agent_orchestrator:
    _type: nim
    model: nvdev/meta/llama-3.1-405b-instruct
    

In [12]:
await agent.prompt("Who was Djikstra?")

/Users/spastoriza/Documents/Programming/public/nat-official/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:715: UserWarning: Model 'nvdev/meta/llama-3.3-70b-instruct' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/Users/spastoriza/Documents/Programming/public/nat-official/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:715: UserWarning: Model 'nvdev/meta/llama-3.3-70b-instruct' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function code_generation_tool by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.


'Edsger Wybe Dijkstra was a Dutch computer scientist, programmer, software engineer, mathematician, and science essayist, best known for his contributions to the development of structured programming languages and his work on the shortest path problem.'